In [26]:
import pandas as pd 
import numpy as np 

In [27]:
df_occupation = pd.read_excel('db_29_1_excel\Occupation Data.xlsx')

In [28]:
df_occupation.head()

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


In [29]:
## Read other files (NOTE: Technology Skills excel file)
df_skills = pd.read_excel('db_29_1_excel\Skills.xlsx')
# df_skills.head()
df_education = pd.read_excel('db_29_1_excel/Education, Training, and Experience.xlsx')
# df_education.tail()
df_work_activities = pd.read_excel('db_29_1_excel\Work Activities.xlsx')
# df_work_activities.head()
df_related_occupations = pd.read_excel('db_29_1_excel\Related Occupations.xlsx')
# df_related_occupations.head()

In [30]:
df_alternate_titles = pd.read_excel('db_29_1_excel\Alternate Titles.xlsx')

In [31]:
df_related_occupations.head()

,O*NET-SOC Code,Title,Related O*NET-SOC Code,Related Title,Relatedness Tier,Index
0,11-1011.00,Chief Executives,11-1021.00,General and Operations Managers,Primary-Short,1
1,11-1011.00,Chief Executives,11-2032.00,Public Relations Managers,Primary-Short,2
2,11-1011.00,Chief Executives,11-9151.00,Social and Community Service Managers,Primary-Short,3
3,11-1011.00,Chief Executives,11-3031.01,Treasurers and Controllers,Primary-Short,4
4,11-1011.00,Chief Executives,11-9199.02,Compliance Managers,Primary-Short,5


In [ ]:
# Read salary information
# df_salary = pd.read_excel('db_29_1_excel\Salary_info.xlsx')

In [ ]:
# df_salary.head()

,OCC_CODE,OCC_TITLE,O_GROUP,H_MEAN,A_MEAN
0,11-1011,Chief Executives,detailed,124.47,258900
1,11-1021,General and Operations Managers,detailed,62.18,129330
2,11-1031,Legislators,detailed,*,68140
3,11-2011,Advertising and Promotions Managers,detailed,73.38,152620
4,11-2021,Marketing Managers,detailed,80,166410


In [ ]:
# df_salary['A_MEAN'].replace({'*':'Not Available','#':'>=239200'},inplace=True)
# df_salary['H_MEAN'].replace({'*':'Not Available','#':'>=115'},inplace=True)

In [ ]:
# df_salary['A_MEAN'].value_counts()

Not Available    3852
>=239200          265
40820              89
44290              88
39820              88
                 ... 
333470              1
363010              1
263940              1
388950              1
154570              1
Name: A_MEAN, Length: 19379, dtype: int64

### Store into ChromaDB

In [11]:
# !pip install chromadb
# import chromadb
# chroma_client = chromadb.Client()
# career_collection = chroma_client.create_collection(name="my_career_collection")
# # Store the Embeddings
# from sentence_transformers import SentenceTransformer
# # Load a Sentence Transformer Model for Embeddings
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

### Merge subgroups in the data

#### 1. Skills

In [32]:
# We have two records for each IM and LV so let us choose one of them
df_skills_filtered = df_skills[df_skills['Scale ID']=='IM'].copy()

In [33]:
df_skills_filtered.head()

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
6,11-1011.00,Chief Executives,2.A.1.d,Speaking,IM,Importance,4.25,8,0.1637,3.9292,4.5708,N,NaN,08/2023,Analyst
8,11-1011.00,Chief Executives,2.A.1.e,Mathematics,IM,Importance,3.25,8,0.2500,2.7600,3.7400,N,NaN,08/2023,Analyst


In [34]:
df_skills_filtered = df_skills_filtered[df_skills_filtered['Data Value'] > 4.0]

In [35]:
df_skills_small = df_skills_filtered[['O*NET-SOC Code','Element Name','Data Value']]

In [36]:
df_skills_small.head()

,O*NET-SOC Code,Element Name,Data Value
0,11-1011.00,Reading Comprehension,4.12
4,11-1011.00,Writing,4.12
6,11-1011.00,Speaking,4.25
12,11-1011.00,Critical Thinking,4.38
20,11-1011.00,Social Perceptiveness,4.12


In [37]:
df_skills_small['O*NET-SOC Code'].nunique()

240

#### 2. Education Training and Experience

In [38]:
# Read the two files on education and get the category in one file
edu_original = pd.read_excel('db_29_1_excel/Education, Training, and Experience.xlsx')
edu_categories = pd.read_excel('db_29_1_excel/Education, Training, and Experience Categories.xlsx')

In [39]:
edu_original.head()

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Category,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Date,Domain Source
0,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),1.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent
1,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),2.0,4.46,28,4.1428,0.6307,25.5524,N,08/2023,Incumbent
2,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),3.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent
3,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),4.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent
4,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),5.0,5.15,28,5.2236,0.6000,32.7756,N,08/2023,Incumbent


In [40]:
edu_categories.tail(5)

,Element ID,Element Name,Scale ID,Scale Name,Category,Category Description
36,3.A.3,On-the-Job Training,OJ,On-The-Job Training (Categories 1-9),5,"Over 6 months, up to and including 1 year"
37,3.A.3,On-the-Job Training,OJ,On-The-Job Training (Categories 1-9),6,"Over 1 year, up to and including 2 years"
38,3.A.3,On-the-Job Training,OJ,On-The-Job Training (Categories 1-9),7,"Over 2 years, up to and including 4 years"
39,3.A.3,On-the-Job Training,OJ,On-The-Job Training (Categories 1-9),8,"Over 4 years, up to and including 10 years"
40,3.A.3,On-the-Job Training,OJ,On-The-Job Training (Categories 1-9),9,Over 10 years


In [41]:
# edu_categories[('Scale ID' == 'PT') & ('Element ID'== '3.A.2')]
edu_categories[(edu_categories['Scale ID'] == 'PT') & (edu_categories['Element ID'] == '3.A.2')]

,Element ID,Element Name,Scale ID,Scale Name,Category,Category Description
23,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),1,None
24,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),2,Up to and including 1 month
25,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),3,"Over 1 month, up to and including 3 months"
26,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),4,"Over 3 months, up to and including 6 months"
27,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),5,"Over 6 months, up to and including 1 year"
28,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),6,"Over 1 year, up to and including 2 years"
29,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),7,"Over 2 years, up to and including 4 years"
30,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),8,"Over 4 years, up to and including 10 years"
31,3.A.2,On-Site or In-Plant Training,PT,On-Site Or In-Plant Training (Categories 1-9),9,Over 10 years


In [42]:
# edu_original['education_description'] = np.where((edu_categories['Scale ID'] == edu_original['Scale ID']) & (edu_categories['Category'] == edu_original['Category']),edu_categories['Category Description'])

In [43]:
# edu_original = edu_original.merge(edu_categories[['Scale ID', 'Category', 'Category Description']], 
#                                   on=['Scale ID', 'Category'], how='left')`

In [46]:
# Merge and populate 'education_description' from 'Category_Description'
edu_original = edu_original.merge(edu_categories[['Scale ID', 'Category', 'Category Description']], 
                                  on=['Scale ID', 'Category'], how='left')

In [47]:
edu_original.columns

Index(['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID',
       'Scale Name', 'Category', 'Data Value', 'N', 'Standard Error',
       'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Date',
       'Domain Source', 'Category Description_x', 'Category Description_y'],
      dtype='object')

In [48]:
edu_original.head(2)

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Category,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Date,Domain Source,Category Description_x,Category Description_y
0,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),1.0,0.00,28,0.0000,NaN,NaN,N,08/2023,Incumbent,Less than a High School Diploma,Less than a High School Diploma
1,11-1011.00,Chief Executives,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),2.0,4.46,28,4.1428,0.6307,25.5524,N,08/2023,Incumbent,High School Diploma - or the equivalent (for e...,High School Diploma - or the equivalent (for e...


In [49]:
edu_filtered = edu_original[['O*NET-SOC Code','Title','Element Name','Element ID','Scale ID','Category','Category Description_x','Data Value']]
# edu_filtered.rename(columns={'Category Description_x':'Description','Element Name': 'Element_Name'},inplace=True)
# EWT - Education, Workforce and Training

In [50]:
edu_filtered.rename(columns={'Category Description_x':'Description','Element Name': 'Element_Name'},inplace=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_13812\4197177563.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edu_filtered.rename(columns={'Category Description_x':'Description','Element Name': 'Element_Name'},inplace=True)


In [51]:
# Education data filtered
edu_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 36209 entries, 0 to 36208
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   O*NET-SOC Code  36209 non-null  object 
 1   Title           36209 non-null  object 
 2   Element_Name    36209 non-null  object 
 3   Element ID      36209 non-null  object 
 4   Scale ID        36209 non-null  object 
 5   Category        35178 non-null  float64
 6   Description     35178 non-null  object 
 7   Data Value      36209 non-null  float64
dtypes: float64(2), object(6)
memory usage: 2.5+ MB


In [52]:
# edu_filtered.drop(edu_filtered['Scale ID']=='IM'
# edu_filtered = edu_filtered.drop(edu_filtered['Scale ID']=='IM')
edu_filtered = edu_filtered[edu_filtered['Scale ID']!='IM']
# edu_filtered.dro

In [53]:
edu_filtered.head(2)

,O*NET-SOC Code,Title,Element_Name,Element ID,Scale ID,Category,Description,Data Value
0,11-1011.00,Chief Executives,Required Level of Education,2.D.1,RL,1.0,Less than a High School Diploma,0.00
1,11-1011.00,Chief Executives,Required Level of Education,2.D.1,RL,2.0,High School Diploma - or the equivalent (for e...,4.46


In [54]:
edu_filtered[edu_filtered['Description'].isnull()]

,O*NET-SOC Code,Title,Element_Name,Element ID,Scale ID,Category,Description,Data Value


#### 4. Work Activities

In [55]:
df_work_activities.head(5)

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,4.A.1.a.1,Getting Information,IM,Importance,4.56,29.0,0.1559,4.2369,4.8756,N,NaN,08/2023,Incumbent
1,11-1011.00,Chief Executives,4.A.1.a.1,Getting Information,LV,Level,4.89,30.0,0.1727,4.5393,5.2458,N,N,08/2023,Incumbent
2,11-1011.00,Chief Executives,4.A.1.a.2,"Monitoring Processes, Materials, or Surroundings",IM,Importance,4.25,30.0,0.2125,3.8130,4.6823,N,NaN,08/2023,Incumbent
3,11-1011.00,Chief Executives,4.A.1.a.2,"Monitoring Processes, Materials, or Surroundings",LV,Level,5.21,30.0,0.3872,4.4133,5.9971,N,N,08/2023,Incumbent
4,11-1011.00,Chief Executives,4.A.1.b.1,"Identifying Objects, Actions, and Events",IM,Importance,4.23,29.0,0.1544,3.9180,4.5507,N,NaN,08/2023,Incumbent


In [56]:
# We have two records for each IM and LV so let us choose one of them
df_work_act_filtered = df_work_activities[df_work_activities['Scale ID']=='IM'].copy()

In [57]:
df_work_act_filtered.head(2)

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,4.A.1.a.1,Getting Information,IM,Importance,4.56,29.0,0.1559,4.2369,4.8756,N,NaN,08/2023,Incumbent
2,11-1011.00,Chief Executives,4.A.1.a.2,"Monitoring Processes, Materials, or Surroundings",IM,Importance,4.25,30.0,0.2125,3.8130,4.6823,N,NaN,08/2023,Incumbent


In [58]:
df_work_act_small = df_work_act_filtered[['O*NET-SOC Code','Element Name','Data Value']]

In [59]:
df_related_occupations.head()

,O*NET-SOC Code,Title,Related O*NET-SOC Code,Related Title,Relatedness Tier,Index
0,11-1011.00,Chief Executives,11-1021.00,General and Operations Managers,Primary-Short,1
1,11-1011.00,Chief Executives,11-2032.00,Public Relations Managers,Primary-Short,2
2,11-1011.00,Chief Executives,11-9151.00,Social and Community Service Managers,Primary-Short,3
3,11-1011.00,Chief Executives,11-3031.01,Treasurers and Controllers,Primary-Short,4
4,11-1011.00,Chief Executives,11-9199.02,Compliance Managers,Primary-Short,5


#### Tools Used:
NOTE: 
1. We use the Commodity Title field as the name of the tool.
2. This will list tools used by each role.


In [60]:
df_tools_used = pd.read_excel("db_29_1_excel\Tools Used.xlsx")
# 'db_29_1_excel\Tools Used.xlsx'

In [61]:
# df_tools_used['O*NET-SOC Code'].nunique()
df_tools_used.rename(columns={'Commodity Title': 'Tools Used'},inplace = True)

#### Technology Skills:
NOTE:
1. Commodity Title


In [62]:
df_tech_skills = pd.read_excel('db_29_1_excel\Technology Skills.xlsx')

In [63]:
df_tech_skills.head(2)

,O*NET-SOC Code,Title,Example,Commodity Code,Commodity Title,Hot Technology,In Demand
0,11-1011.00,Chief Executives,Adobe Acrobat,43232202,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,43232306,Data base user interface and query software,N,N


In [64]:
df_tech_skills.rename(columns={'Example':'Technological_skill'},inplace=True)

#### Task Statements
1. This file contains information on the tasks associated to a specific role.
2. The tasks are listed for each occupation.

In [65]:
df_tasks = pd.read_excel('db_29_1_excel\Task Statements.xlsx')

In [66]:
df_tasks.head(2)

,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent


In [68]:
# import pandas as pd
# #### Work Context
# df_work_context = pd.read_excel('db_29_1_excel\Work Context.xlsx')

In [ ]:
# df_work_context.head(2)

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Category,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,4.C.1.a.2.c,Public Speaking,CX,Context,NaN,3.07,37.0,0.2851,2.4923,3.6486,N,NaN,08/2023,Incumbent
1,11-1011.00,Chief Executives,4.C.1.a.2.c,Public Speaking,CXP,Context (Categories 1-5),1.0,0.13,37.0,0.1370,0.0160,1.0770,N,NaN,08/2023,Incumbent


In [ ]:
# work_context_filtered = df_work_context[(df_work_context['Scale ID']=='CX') | (df_work_context['Scale ID']=='CT')]

In [ ]:
# work_context_filtered.info()
# work_context_filtered.to_excel('Updated_Work_Context.xlsx')

In [70]:
career_data = {}
for index, row in df_occupation.iterrows():
    soc_code = row["O*NET-SOC Code"]
    
    # Define mapping for Scale ID renaming
    scale_id_mapping = {
        "RL": "Education Level",
        "RW": "Work Experience Level",
        "PT": "On-site Plant Training",
        "OJ": "On the Job Training"
    }

    # Filter education data for the current SOC code
    edu_data = edu_filtered[edu_filtered["O*NET-SOC Code"] == soc_code].copy()

    # Apply renaming based on Scale ID
    edu_data["Field Name"] = edu_data["Scale ID"].map(scale_id_mapping)  # Create a new column for dynamic naming
    edu_data.rename(columns={"Description": "Temp_Description"}, inplace=True)  # Temporarily rename Description
    
    # Construct final dictionary format
    # education_list = [
    #     {row["Field Name"]: row["Temp_Description"], "Importance Level": row["Data Value"]}
    #     for _, row in edu_data.iterrows() if row["Field Name"] is not None  # Exclude unmatched Scale IDs
    # ]
    education_list= []
    work_experience_list = []
    on_job_training_list = []

    for _, edu_row in edu_data.iterrows():
        if edu_row["Field Name"] == "Education Level":
            education_list.append({"Education Level": edu_row["Temp_Description"], "Importance Level": edu_row["Data Value"]})
        elif edu_row["Field Name"] == "Work Experience Level":
            work_experience_list.append({"Work Experience Level": edu_row["Temp_Description"], "Importance Level": edu_row["Data Value"]})
        elif edu_row["Field Name"] in ["On-site Plant Training", "On the Job Training"]:
            on_job_training_list.append({"On Job Training": edu_row["Temp_Description"], "Importance Level": edu_row["Data Value"]})

    # Filter tasks for the given SOC code
    tasks_data = df_tasks[df_tasks["O*NET-SOC Code"] == soc_code]
    core_tasks = tasks_data[tasks_data["Task Type"] == "Core"]["Task"].tolist()
    supplemental_tasks = tasks_data[tasks_data["Task Type"] == "Supplemental"]["Task"].tolist()
    # Build the dictionary for the given occupation
    career_data[soc_code] = {
        "job_title": row["Title"],
        "job_description": row["Description"],
        "skills": df_skills_small[df_skills_small["O*NET-SOC Code"] == soc_code][["Element Name", "Data Value"]]
                   .rename(columns={"Element Name": "Soft Skills", "Data Value": "Importance"})
                   .to_dict(orient="records"),
        # "education": education_list,
        "education": education_list,
        "work_experience": work_experience_list,
        "on_job_training": on_job_training_list,
        "work_activities": df_work_act_small[df_work_act_small["O*NET-SOC Code"] == soc_code][["Element Name", "Data Value"]]
                           .rename(columns={"Element Name": "Work Activity", "Data Value": "Importance"})
                           .to_dict(orient="records"),
        # 'salary': df_salary[df_salary["O*NET-SOC Code"] == soc_code ],
        # 'salary': df_salary[df_salary["OCC_CODE"] == soc_code[:7]]
        "alternate_titles": df_alternate_titles[df_alternate_titles["O*NET-SOC Code"] == soc_code]['Alternate Title']
                            .values.tolist(),
        # "related_jobs": df_related_occupations[df_related_occupations["O*NET-SOC Code"] == soc_code][["Related O*NET-SOC Code", "Related Title"]]
                        # .to_dict(orient="records"),
        "tools_used": df_tools_used[df_tools_used["O*NET-SOC Code"]==soc_code]['Tools Used'].values.tolist(),
        "technical_skills": df_tech_skills[df_tech_skills["O*NET-SOC Code"]==soc_code]['Technological_skill'].values.tolist(),
        # "job_tasks": df_tasks[df_tasks["O*NET-SOC Code"]==soc_code]['Task'].values.tolist()
        "core_tasks": core_tasks,
        "supplemental_tasks": supplemental_tasks
    }


In [2]:
# career_data["17-2081.00"]['skills']['Importance']

In [1]:
# df_salary[df_salary["OCC_CODE"] == '17-2081.00'[:7]]

In [71]:
import json
with open("career_data_attempt0203.json", "w") as f:
    json.dump(career_data, f, indent=4)